# Etude No. 1: Omission Visualization Gallery
This interactive notebook contains a detailed showcase for each of the 16 canonical visualization tasks in the `jnwb` gallery. Running this notebook reproduces all the figures and reports.


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pynwb

# Ensure local jnwb is imported
sys.path.insert(0, os.path.abspath('.'))
import jnwb as oa
import jnwb.visual_qc as qc
import jnwb.viz as viz

# Global matplotlib styling
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

NWB_PATH = "D:/analysis/nwb/sub-C31o_ses-230630_rec.nwb"
BASE_OUT = "outputs/visualization_gallery"
os.makedirs(BASE_OUT, exist_ok=True)

def save_path(task_name, filename):
    folder = os.path.join(BASE_OUT, task_name)
    os.makedirs(folder, exist_ok=True)
    return os.path.join(folder, filename)

# Load the session
sess = oa.read(NWB_PATH)

# Setup synthetic TFR fallback for loading preprocessed TFR arrays (if they do not exist)
def _synth_tfr(seed=0, channels=64, freqs=128, time=100, trials=20):
    rng = np.random.default_rng(seed)
    base = rng.standard_normal((channels, freqs, time, trials))
    for fi, (lo, hi) in enumerate([(1,4),(4,8),(8,15),(15,30),(30,60),(60,120)]):
        band_slice = slice(int(lo * freqs / 150), int(hi * freqs / 150))
        base[:, band_slice, :, :] += rng.standard_normal((channels, band_slice.stop - band_slice.start, time, trials)) * 0.5
    return base.astype(np.float32)

orig_tfr = sess.tfr_from_preprocessed
def _tfr_mock(area, band=None, condition=None, tfr_dir=None):
    res = orig_tfr(area, band, condition, tfr_dir)
    return res if res is not None else _synth_tfr()
sess.tfr_from_preprocessed = _tfr_mock

# Select a high-FR unit for showcases
high_fr_units = sess.find_single_units(firing_rate_range=(10.0, 500.0))
target_unit = float(high_fr_units.iloc[0]['unit_id']) if len(high_fr_units) > 0 else 2.0
all_unit_ids = high_fr_units['unit_id'].astype(float).tolist()[:8]
print(f"Loaded session. Selected unit: {target_unit}")


## Task 1: Single Unit Raster Suite
Generates a 3-panel figure showing the Spike Raster, PSTH (with baseline), and Autocorrelogram.


In [ ]:
res = sess.raster_suite(target_unit)
fig = res["figure"]
fig.savefig(save_path("task_01_raster", "raster_suite.png"), bbox_inches="tight")
plt.show()


## Task 2: Raw LFP Traces (Probe B / 1)
Plots raw LFP time-series for channels 44, 47, and 50 of Probe B.


In [ ]:
N_SAMP = 10000
CH = [44, 47, 50]
labels = [f"Ch {c}" for c in CH]
colors = ["#CFB87C", "#00BBAA", "#7B2FBE"]

with pynwb.NWBHDF5IO(NWB_PATH, "r", load_namespaces=True) as io:
    nwb = io.read()
    lfp_obj = nwb.acquisition["probe_1_lfp"]
    rate = getattr(lfp_obj, "rate", 1000.0) or 1000.0
    lfp = lfp_obj.data[:N_SAMP, CH]

t_ms = np.arange(N_SAMP) / rate * 1000
fig, axes = plt.subplots(len(CH), 1, figsize=(12, 6), sharex=True)
for ax, data, label, color in zip(axes, lfp.T, labels, colors):
    ax.plot(t_ms, data, color=color, linewidth=0.8)
    ax.set_ylabel(label, rotation=0, labelpad=40, va="center")
    ax.set_yticks([])
axes[-1].set_xlabel("Time (ms)")
fig.suptitle("LFP Traces — Probe B · Channels 44, 47, 50", fontweight="bold")
fig.tight_layout()
fig.savefig(save_path("task_02_lfp", "lfp_probeB_ch44_47_50.png"), bbox_inches="tight")
plt.show()


## Task 3: Raw MUAe Traces (Probe A / 0)
Plots raw MUA envelope traces for channels 1 and 127 of Probe A.


In [ ]:
CH_MUA = [1, 127]
with pynwb.NWBHDF5IO(NWB_PATH, "r", load_namespaces=True) as io:
    nwb = io.read()
    mobj = nwb.acquisition["probe_0_muae"]
    mrate = getattr(mobj, "rate", 1000.0) or 1000.0
    muae = mobj.data[:N_SAMP, CH_MUA]

t_ms = np.arange(N_SAMP) / mrate * 1000
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
for ax, data, ch, color in zip(axes, muae.T, CH_MUA, ["#E05C3A", "#3A6EA5"]):
    ax.fill_between(t_ms, 0, data, alpha=0.55, color=color)
    ax.plot(t_ms, data, color=color, linewidth=0.7)
    ax.set_ylabel(f"Ch {ch}", rotation=0, labelpad=35, va="center")
    ax.set_yticks([])
axes[-1].set_xlabel("Time (ms)")
fig.suptitle("MUAe Traces — Probe A · Channels 1, 127", fontweight="bold")
fig.tight_layout()
fig.savefig(save_path("task_03_muae", "muae_probeA_ch1_127.png"), bbox_inches="tight")
plt.show()


## Task 4: Single Channel TFR Image
Generates a 2D Log-Frequency spectrogram image for channel 22, Probe A (PFC) in condition AAAB.


In [ ]:
tfr = sess.tfr_from_preprocessed(area="PFC", condition="AAAB")
ch22 = tfr[22, :, :, 0]
baseline = np.mean(ch22[:, :20], axis=1, keepdims=True)
ch22_db = ch22 - baseline
n_f, n_t = ch22_db.shape
freqs_ax = np.linspace(1, 150, n_f)
times_ax = np.linspace(-1000, 2000, n_t)

fig, ax = plt.subplots(figsize=(11, 5))
vmax = np.percentile(np.abs(ch22_db), 97)
im = ax.pcolormesh(times_ax, freqs_ax, ch22_db, shading="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
ax.set_yscale("log")
ax.set_yticks([4, 8, 15, 30, 60, 100, 150])
ax.get_yaxis().set_major_formatter(plt.ScalarFormatter())
ax.axvline(0, color="k", linestyle="--", linewidth=1.0, alpha=0.6)
fig.colorbar(im, ax=ax, label="Power (dB re baseline)")
ax.set_xlabel("Time from stimulus onset (ms)")
ax.set_ylabel("Frequency (Hz)")
ax.set_title("TFR — Probe A · Channel 22 · PFC · Condition AAAB")
fig.tight_layout()
fig.savefig(save_path("task_04_tfr_image", "tfr_ch22_probeA.png"), bbox_inches="tight")
plt.show()


## Task 5: TFR Band Traces
Averages and plots power across channels 20–80 for the 7 canonical frequency bands.


In [ ]:
tfr_all = sess.tfr_from_preprocessed(area="PFC", condition="AAAB")
tfr_sub = tfr_all[20:80, :, :, :]
n_f_full = tfr_sub.shape[1]
times_ax = np.linspace(-1000, 2000, tfr_sub.shape[2])

BAND_RANGES = {
    "delta": (1, 4), "theta": (4, 8), "alpha": (8, 15), "beta": (15, 30),
    "low_gamma": (30, 60), "high_gamma": (60, 120), "broadband": (1, 150)
}
BAND_COLORS = {
    "delta": "#4477AA", "theta": "#66CCEE", "alpha": "#228833", "beta": "#CCBB44",
    "low_gamma": "#EE6677", "high_gamma": "#AA3377", "broadband": "#BBBBBB"
}

fig, ax = plt.subplots(figsize=(12, 5))
for band, (lo, hi) in BAND_RANGES.items():
    fi_lo = max(0, int(lo * n_f_full / 150))
    fi_hi = min(n_f_full, int(hi * n_f_full / 150))
    power = np.mean(tfr_sub[:, fi_lo:fi_hi, :, :], axis=(0, 1, 3))
    ax.plot(times_ax, power, color=BAND_COLORS[band], label=band, alpha=0.85)

ax.axvline(0, color="k", linestyle="--", linewidth=1.0, alpha=0.5)
ax.set_xlabel("Time from stimulus onset (ms)")
ax.set_ylabel("Mean Power (a.u.)")
ax.set_title("TFR Band Traces — Channels 20–80 · PFC · AAAB")
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig(save_path("task_05_tfr_band_traces", "tfr_band_traces_ch20_80.png"), bbox_inches="tight")
plt.show()


## Task 6: Unit Quality Distribution
Plots distributions of firing rates, SNR, waveform durations, and stability categories.


In [ ]:
if sess._units_df is not None:
    fig = qc.plot_unit_quality_distribution(sess._units_df)
    fig.savefig(save_path("task_06_quality_distribution", "unit_quality_distribution.png"), bbox_inches="tight")
    plt.show()


## Task 7: Noise vs. Signal Scatter
Plots multi-metric scatter tradeoffs across units.


In [ ]:
if sess._units_df is not None:
    fig = qc.plot_noise_vs_signal(sess._units_df)
    fig.savefig(save_path("task_07_noise_vs_signal", "noise_vs_signal.png"), bbox_inches="tight")
    plt.show()


## Task 8: Pie Charts of Unit Quality
Displays unit stability categories grouped by recording area.


In [ ]:
res_pie = sess.pie_charts(criteria=None, by_area=True)
if isinstance(res_pie, dict) and "figures" in res_pie and res_pie["figures"]:
    for name, fig in res_pie["figures"].items():
        fig.savefig(save_path("task_08_pie_charts", f"pie_{name}.png"), bbox_inches="tight")
        print(f"Saved pie chart for area: {name}")


## Task 9: plot_tfr (PFC, AAXB, Phase 3)
Plots a baseline-subtracted Log-Frequency TFR for PFC in AAXB condition.


In [ ]:
res = sess.plot_tfr(area="PFC", condition="AAXB", phase=3)
res["figure"].savefig(save_path("task_09_plot_tfr", "tfr_PFC_AAXB_phase3.png"), bbox_inches="tight")
plt.show()


## Task 10: Trial-Averaged Spectrogram (MT, AAXB)
Heatmap of trial-averaged spectral power for the MT area in AAXB condition.


In [ ]:
res = sess.trial_averaged_plot(area="MT", phase=2, condition="AAXB")
res["figure"].savefig(save_path("task_10_trial_averaged", "trial_averaged_MT_AAXB.png"), bbox_inches="tight")
plt.show()


## Task 11: Channel-Averaged Power Spectrum
1D Power Spectral Density (PSD) line plot for MT averaged across channels and trials.


In [ ]:
res = sess.channel_averaged_plot(area="MT", phase=2, condition="AAXB")
res["figure"].savefig(save_path("task_11_channel_averaged", "channel_averaged_MT.png"), bbox_inches="tight")
plt.show()


## Task 12: Spectrolaminar Motif (V4, AAAB)
Performs layer-wise spectral analysis for V4, saving JSON and a superficial vs. deep layer heatmap.


In [ ]:
res_sl = sess.spectrolaminar_motif(area="V4", condition="AAAB")
if "layer_data" in res_sl:
    ld = res_sl["layer_data"]
    layers = list(ld.keys())
    arrays = [np.atleast_2d(np.array(ld[k])) for k in layers]
    fig, axes = plt.subplots(1, len(layers), figsize=(6 * len(layers), 4))
    if len(layers) == 1:
        axes = [axes]
    for ax, arr, lname in zip(axes, arrays, layers):
        arr2d = arr if arr.ndim == 2 else arr.reshape(arr.shape[0], -1)
        im = ax.imshow(arr2d, aspect="auto", origin="lower", cmap="viridis", interpolation="nearest")
        fig.colorbar(im, ax=ax, label="Power")
        ax.set_title(f"{lname.capitalize()} Layer")
        ax.set_xlabel("Time Bins")
        ax.set_ylabel("Frequency Bins")
    fig.suptitle("Spectrolaminar Motif — V4 · AAAB", fontweight="bold")
    fig.tight_layout()
    fig.savefig(save_path("task_12_spectrolaminar", "spectrolaminar_heatmap.png"), bbox_inches="tight")
    plt.show()


## Task 13: LFP TFR Trace Suite (V1 Deep)
Generates the publication-grade 2-row aligned spectrogram trace suite for a deep layer of area V1.


In [ ]:
res = sess.lfp_tfr_trace_suite_omission(area="V1", layer="deep")
res["figure"].savefig(save_path("task_13_trace_suite", "lfp_tfr_trace_suite_V1.png"), bbox_inches="tight")
plt.show()


## Task 14: LFP Inter-Area Correlation Heatmap
Computes and plots a 22x22 correlation matrix of LFP band power averages across all area-layers for the Alpha band.


In [ ]:
res = sess.lfp_tfr_trace_correlation(band_name="Alpha")
res["figure"].savefig(save_path("task_14_correlation", "lfp_correlation_alpha.png"), bbox_inches="tight")
plt.show()


## Task 15: Condition-Family Raster Grid
Generates a multi-panel grid of spike rasters for a set of high-FR units grouped by condition family A.


In [ ]:
figs_list = viz.raster_grid_by_family(sess, unit_ids=all_unit_ids, family="A")
if not isinstance(figs_list, list):
    figs_list = [figs_list]
for page_i, fig in enumerate(figs_list):
    fig.savefig(save_path("task_15_raster_grid", f"raster_grid_A_page{page_i+1}.png"), bbox_inches="tight")
    fig.show()


## Task 16: Creative Spectral Radar & Granger Network
Plots a polar radar map representing relative band powers and a directional lag network heatmap representing leads and lags.


In [ ]:
AREAS = ["V1", "V4", "MT", "MST", "PFC"]
BANDS_SHORT = ["delta", "theta", "alpha", "beta", "low_gamma"]

fig = plt.figure(figsize=(16, 6))

# Left: Radar chart
ax_radar = fig.add_subplot(121, polar=True)
angles = np.linspace(0, 2 * np.pi, len(BANDS_SHORT), endpoint=False).tolist()
angles += angles[:1]
ax_radar.set_thetagrids(np.degrees(angles[:-1]), BANDS_SHORT)
colors_r = ["#CFB87C", "#00BBAA", "#7B2FBE", "#E05C3A", "#3A6EA5"]

for area, color in zip(AREAS, colors_r):
    tfr_a = sess.tfr_from_preprocessed(area=area, condition="AAAB")
    values = []
    for band, (lo, hi) in list(BAND_RANGES.items())[:5]:
        fi_lo = max(0, int(lo * tfr_a.shape[1] / 150))
        fi_hi = min(tfr_a.shape[1], int(hi * tfr_a.shape[1] / 150))
        power = float(np.mean(tfr_a[:, fi_lo:fi_hi, :, :]))
        values.append(power)
    vmax_local = max(values) if max(values) > 0 else 1.0
    vals_norm = [v / vmax_local for v in values] + [values[0] / vmax_local]
    ax_radar.plot(angles, vals_norm, color=color, linewidth=1.8, label=area)
    ax_radar.fill(angles, vals_norm, color=color, alpha=0.1)

ax_radar.set_title("Multi-Area Band-Power Radar", pad=18, fontweight="bold")
ax_radar.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))

# Right: Directional lag heatmap
ax_net = fig.add_subplot(122)
n = len(AREAS)
granger_proxy = np.zeros((n, n))

for i, a1 in enumerate(AREAS):
    t1 = sess.tfr_from_preprocessed(area=a1, condition="AAAB")
    s1 = np.mean(t1[:, :, :, :], axis=(0, 1, 3))
    for j, a2 in enumerate(AREAS):
        if i != j:
            t2 = sess.tfr_from_preprocessed(area=a2, condition="AAAB")
            s2 = np.mean(t2[:, :, :, :], axis=(0, 1, 3))
            cc = np.correlate(s1 - s1.mean(), s2 - s2.mean(), mode="full")
            lag = int(np.argmax(cc) - len(s1) + 1)
            granger_proxy[i, j] = lag

im = ax_net.imshow(granger_proxy, cmap="RdBu_r", vmin=-np.abs(granger_proxy).max(), vmax=np.abs(granger_proxy).max())
ax_net.set_xticks(range(n)); ax_net.set_xticklabels(AREAS)
ax_net.set_yticks(range(n)); ax_net.set_yticklabels(AREAS)
ax_net.set_title("Directionality Proxy (Cross-Corr Peak Lag)", fontweight="bold")
ax_net.set_xlabel("Target Area")
ax_net.set_ylabel("Source Area")

for i in range(n):
    for j in range(n):
        ax_net.text(j, i, f"{granger_proxy[i,j]:+.0f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(granger_proxy[i,j]) > granger_proxy.max() * 0.5 else "black")

fig.colorbar(im, ax=ax_net, label="Lag (time bins; + = row leads)")
fig.suptitle("Creative — Multi-Area Spectral Comparison & Directionality", fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(save_path("task_16_creative", "multi_area_radar_and_network.png"), bbox_inches="tight")
plt.show()
